# 02 — CNN from Scratch

Train a custom CNN with 5 convolutional blocks.
- Conv2d → BN → ReLU → Conv2d → BN → ReLU → MaxPool (×5)
- Global Average Pooling → FC classifier
- Data augmentation: flip, rotation, jitter, crop

### Expected Output
- Best checkpoint → `ml/artifacts/checkpoints/cnn_best.pth`
- Training curves, confusion matrix, per-class F1
- Classification report and training summary JSON

In [ ]:
import sys, os
from pathlib import Path

PROJECT_ROOT = Path(os.getcwd()).resolve()
if 'notebooks' in str(PROJECT_ROOT):
    PROJECT_ROOT = PROJECT_ROOT.parent.parent
sys.path.insert(0, str(PROJECT_ROOT))
print(f'Project root: {PROJECT_ROOT}')

In [ ]:
from ml.src.utils.seed import set_seed
from ml.src.utils.device import get_device
from ml.src.utils.io import load_config
from ml.src.utils.manifests import create_dataloaders
from ml.src.data.transforms import get_train_transforms, get_eval_transforms
from ml.src.models.cnn import CattleCNN
from ml.src.training.trainer import Trainer

set_seed(42)
device = get_device()

## 1. Load Config & Data

In [ ]:
config = load_config('cnn', PROJECT_ROOT / 'ml' / 'configs')
config['num_classes'] = 26

img_size = config['image']['size']
batch_size = config['training']['batch_size']

print(f'Image size: {img_size}')
print(f'Batch size: {batch_size}')
print(f'Conv channels: {config["model"]["architecture"]["conv_channels"]}')
print(f'Epochs: {config["training"]["num_epochs"]}')

In [ ]:
train_transforms = get_train_transforms(img_size=img_size)
eval_transforms = get_eval_transforms(img_size=img_size)

dataloaders = create_dataloaders(
    manifests_dir=PROJECT_ROOT / 'ml' / 'artifacts' / 'manifests',
    data_root=PROJECT_ROOT / 'Cattle_Resized',
    train_transform=train_transforms,
    eval_transform=eval_transforms,
    batch_size=batch_size,
    num_workers=config['data'].get('num_workers', 4),
)

class_names = dataloaders['train'].dataset.classes
print(f'Classes: {len(class_names)}')
print(f'Train: {len(dataloaders["train"].dataset)} | Val: {len(dataloaders["val"].dataset)} | Test: {len(dataloaders["test"].dataset)}')

## 2. Create Model

In [ ]:
model = CattleCNN.from_config(config)
print(model)

total_params = sum(p.numel() for p in model.parameters())
print(f'\nTotal parameters: {total_params:,}')
print(f'Model size: {total_params * 4 / 1024 / 1024:.1f} MB (float32)')

## 3. Train

In [ ]:
trainer = Trainer(
    model=model,
    config=config,
    dataloaders=dataloaders,
    class_names=class_names,
    device=device,
    model_name='cnn',
)

training_summary = trainer.train()

## 4. Evaluate on Test Set

In [ ]:
test_results = trainer.evaluate(split='test')

## 5. Save Artifacts

In [ ]:
trainer.save_artifacts()

print('\n=== CNN from Scratch Summary ===')
print(f'Best Val Loss:  {training_summary["best_val_loss"]:.4f}')
print(f'Best Val Acc:   {training_summary["best_val_accuracy"]:.4f}')
print(f'Test Accuracy:  {test_results["metrics"]["accuracy"]:.4f}')
print(f'Test Macro F1:  {test_results["metrics"]["macro_f1"]:.4f}')
print(f'Latency:        {test_results["latency"]["avg_ms"]:.2f} ms')
print(f'Model Size:     {test_results["model_size_mb"]:.2f} MB')

## 6. Classification Report

In [ ]:
print(test_results['metrics']['classification_report'])

---
**✅ CNN from Scratch complete.** Proceed to `03_resnet_transfer_learning.ipynb`.